In [5]:
#pip install xskillscore

In [6]:
#pip install  climetlab

In [7]:
#pip install --upgrade fair-research-login 

In [8]:
#pip install geocat.viz

In [9]:
#conda install -c conda-forge xcdat

In [10]:
#pip install numpy

In [11]:
#pip install pyproj

In [12]:
#pip install ESMPy

In [6]:
#pip install scikit-image

In [9]:
#pip install climetlab

In [1]:
import os
import collections
import xarray as xr
import numpy as np
import pandas as pd
import metpy.calc as mpcalc
import xcdat as xc
import xskillscore as xs

from datetime import datetime
from skimage.feature import peak_local_max
import cartopy.crs as ccrs
from scipy.stats import pearsonr
from scipy.signal import butter, filtfilt, sosfilt,lfilter
import datetime as dt

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
from matplotlib.patches import Polygon
from matplotlib import ticker

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker

#import climetlab as cml

import cmaps as gvcmaps
import geocat.viz.util as gvutil
import geocat.viz as gv
import warnings
warnings.filterwarnings("ignore")


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [3]:
def read_dart_obs_diag(fname, ObsTypeString, region, CopyString, QCString, verbose): 
    # read_obs_netcdf reads in the netcdf flavor observation sequence file
    #and returns a subsetted structure.
    #
    # fname         = 'obs_epoch_001.nc';
    # ObsTypeString = 'RADIOSONDE_U_WIND_COMPONENT';   % or 'ALL' ...
    # region        = [0 360 -90 90 -Inf Inf];
    # CopyString    = 'NCEP BUFR observation';         % or 'ALL' ...
    # QCString      = 'DART quality control';
    # verbose       = 1;   % anything > 0 == 'true'
    # obs = read_obs_netcdf(fname, ObsTypeString, region, CopyString, QCString, verbose);
    #
    # The return variable 'obs' is a structure. As an example ...
    #
    #           fname: 'obs_epoch_001.nc'
    #   ObsTypeString: 'RADIOSONDE_U_WIND_COMPONENT'
    #          region: [0 360 -90 90 -Inf Inf]
    #      CopyString: 'NCEP BUFR observation'
    #        QCString: 'DART quality control'
    #         verbose: 1
    #      timestring: [2x20 char]
    #            lons: [2343x1 double]
    #            lats: [2343x1 double]
    #               z: [2343x1 double]
    #             obs: [2343x1 double]
    #            Ztyp: [2343x1 int32]
    #            keys: [2343x1 int32]
    #            time: [2343x1 double]
    #              qc: [2343x1 int32]

    #Satellite    = (/"GPSRO", "SAT", "AIRS"/)
    #NCEP_PREBUFR = (/"AIRCRAFT", "ACARS","RADIOSONDE","MARINE"/)
    
    rpath = os.path.join(data_dir,fname)
    dr = xc.open_dataset(rpath)

    f = addfile(fname,"r")
    time = f->time
    QCStrings = chartostring(f->QCMetaData)
    CopyStrings = chartostring(f->CopyMetaData)     # (copy,string)
    ObsTypeStrings = chartostring(f->ObsTypesMetaData) # (ObsTypes,string)
    ObsTypes = f->ObsTypes         # (ObsTypes)
    ObsIndex = f->ObsIndex         # (ObsIndex) i.e. time
    obs_type = f->obs_type
    obs_keys = f->obs_keys

    if(isfilevar(f,"which_vert"))then 
      z_type         = f->which_vert  
    else 
      z_type         = new(dimsizes(obs_keys),integer)
      z_type         = 1
    end if              
   
    z_pos            = tostring(z_type)
    z_pos            = where(z_type.eq.-2,"up",z_pos)
    z_pos            = where(z_type.eq.-1,"up",z_pos)
    z_pos            = where(z_type.eq.1,"down",z_pos)
    z_pos            = where(z_type.eq.2,"down",z_pos)
    z_pos            = where(z_type.eq.3,"up",z_pos)
    z_unt            = tostring(z_type)
    z_unt            = where(z_type.eq.-2,"unknown",z_unt)
    z_unt            = where(z_type.eq.-1,"surface",z_unt)
    z_unt            = where(z_type.eq.1,"model level",z_unt)
    z_unt            = where(z_type.eq.2,"pressure",z_unt)
    z_unt            = where(z_type.eq.3,"height",z_unt)
  
    loc              = f->location
    obs              = f->observations
    qc               = f->qc

    if ( verbose ) then
      my_types  = get_unique_values(obs_type)
      do i = 0,dimsizes(my_types)-1,1
         obtype = my_types(i) 
         print(obtype)
         inds   = ind(obs_type.eq.obtype) 
         zall   = loc(inds,2) 
         myz    = get_unique_values(zall)    
         print("")
         print("N = " +num(.not.ismissing(myz)) +" between levels " + min(myz) + " and " + max(myz)  + \
               " for " + ObsTypeStrings(obtype))
         print("")
         delete([/obtype,inds,zall,myz/])
      end do 
      delete(my_types)
    end if 

    ;;extract the location information;;;;
    if(str_squeeze(CopyString).ne." ")then 
      icopy = ind(str_squeeze(CopyStrings).eq.str_squeeze(CopyString))
      obs1  = obs(:,icopy) 
    else
      print("") 
      print("The required observation data "+ CopyString + " does not exit, please check your data!!")
      print("")
      exit 
    end if  

    #extract quality control information;;;;;
    if(str_squeeze(QCString).ne." ")then 
      iqc = ind(str_squeeze(QCStrings).eq.str_squeeze(QCString))
      qc1 = qc(:,iqc)
    else
      qc1 = qc(:,0)
    end if 
    delete([/qc,obs/])  

   ;;extract data in the specified region;;
   regs   = tofloat(str_split(region, " "))
   indreg = ind ( (loc(:,0).ge.regs(0).and.loc(:,0).le.regs(1)) .and. \
                  (loc(:,1).ge.regs(2).and.loc(:,1).le.regs(3)) .and. \
                  (loc(:,2).ge.regs(4).and.loc(:,2).le.regs(5)) )
   if(.not.all(ismissing(indreg)))then 
     loc2  = loc(indreg,:)
     qc2   = qc1(indreg)  
     obs2  = obs1(indreg)
     oind2 = ObsIndex(indreg)
     ztyp2 = z_type(indreg)
     zpos2 = z_pos(indreg)
     zunt2 = z_unt(indreg)
     time2 = time(indreg)
     otyp2 = obs_type(indreg)
     okey2 = obs_keys(indreg)
   else
     loc2  = loc
     qc2   = qc1
     obs2  = obs1
     oind2 = ObsIndex
     ztyp2 = z_type
     zpos2 = z_pos
     zunt2 = z_unt
     time2 = time
     otyp2 = obs_type
     okey2 = obs_keys
   end if 
   delete([/loc,qc1,obs1,indreg,z_type,z_pos,z_unt,time/])
  
   #extract certain type of data
   if(str_lower(str_squeeze(ObsTypeString)).ne."all")then
     iobtyp = ind(str_squeeze(ObsTypeStrings).eq.str_squeeze(ObsTypeString))
     print("")
     print("only process this type of data: "+ ObsTypeStrings(iobtyp))
     print("")
     indtyp = ind(otyp2.eq.ObsTypes(iobtyp))
     ostr1  = ObsTypes(iobtyp)
     ostr2  = ObsTypeStrings(iobtyp)
   else
     indtyp = ind(.not.ismissing(otyp2))
     myotyp = get_unique_values(otyp2)
     ind1d  = get1Dindex(ObsTypes,myotyp)
     ostr1  = ObsTypes(ind1d)
     ostr2  = ObsTypeStrings(ind1d)
     delete([/myotyp,ind1d/])
   end if
 
   dart_obs        = obs2(indtyp)
   dart_obs@numobs = num(.not.ismissing(indtyp))
   dart_obs@lon    = loc2(indtyp,0)
   dart_obs@lat    = loc2(indtyp,1)
   dart_obs@lev    = loc2(indtyp,2)
   dart_obs@ztyp   = ztyp2(indtyp)
   dart_obs@zpos   = zpos2(indtyp) 
   dart_obs@zunt   = zunt2(indtyp)
   dart_obs@time   = time2(indtyp) 
   dart_obs@qc     = qc2(indtyp)
   dart_obs@oname1 = ostr1
   dart_obs@oname2 = ostr2
   dart_obs@otype  = otyp2(indtyp) 
   dart_obs@okey   = okey2(indtyp)
   dart_obs@oindex = oind2(indtyp)
   delete([/obs2,loc2,qc2,indtyp,ztyp2,zpos2,zunt2,time2,otyp2,okey2,ostr1,ostr2,oind2/])
   delete([/ObsIndex,QCStrings,CopyStrings,ObsTypeStrings,ObsTypes,obs_type,obs_keys/])
   
   return(dart_obs)


IndentationError: unindent does not match any outer indentation level (<string>, line 108)

In [8]:
def extract_exp_info():
    exp_dict  = {
                 'DARTEN10': {
                             'run': 'DARTEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en10',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
                 'DARTEN20': {
                             'run': 'DARTEN20_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en20',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
                 'DARTEN40': {
                             'run': 'DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en40',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
                 'DARTEN80': {
                             'run': 'DARTEN80_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy',
                             'key': 'dart_en80',
                             'diag1': 'obs_seq',
                             'diag2': 'obs_diag',
                             'diag3': 'closest_member'
                 },
    }

    return exp_dict


In [126]:
def read_dart_obs_diag(var,var_dict,date,path,file): 
    # read_obs_netcdf reads in the netcdf flavor observation sequence file
    #and returns a subsetted structure.
    #
    # fname         = 'obs_epoch_001.nc';
    # ObsTypeString = 'RADIOSONDE_U_WIND_COMPONENT';   % or 'ALL' ...
    # region        = [0 360 -90 90 -Inf Inf];
    # CopyString    = 'NCEP BUFR observation';         % or 'ALL' ...
    # QCString      = 'DART quality control';
    # verbose       = 1;   % anything > 0 == 'true'
    # obs = read_obs_netcdf(fname, ObsTypeString, region, CopyString, QCString, verbose);
    #
    # The return variable 'obs' is a structure. As an example ...
    #
    #           fname: 'obs_epoch_001.nc'
    #   ObsTypeString: 'RADIOSONDE_U_WIND_COMPONENT'
    #          region: [0 360 -90 90 -Inf Inf]
    #      CopyString: 'NCEP BUFR observation'
    #        QCString: 'DART quality control'
    #         verbose: 1
    #      timestring: [2x20 char]
    #            lons: [2343x1 double]
    #            lats: [2343x1 double]
    #               z: [2343x1 double]
    #             obs: [2343x1 double]
    #            Ztyp: [2343x1 int32]
    #            keys: [2343x1 int32]
    #            time: [2343x1 double]
    #              qc: [2343x1 int32]

    #Satellite    = (/"GPSRO", "SAT", "AIRS"/)
    #NCEP_PREBUFR = (/"AIRCRAFT", "ACARS","RADIOSONDE","MARINE"/)
    
    rpath = os.path.join(path,file)
    #print(rpath)
    #print(xxx)
    #dr = xc.open_dataset(rpath)
    dr = xc.open_mfdataset(rpath,decode_times=False)
    #print(dr)
    #print(xxx)
    
    ObsTypesMetaData = np.array([char.decode('utf-8').strip() for char in dr['ObsTypesMetaData'].values])
    QCMetaData       = np.array([char.decode('utf-8').strip() for char in dr['QCMetaData'].values])
    CopyMetaData     = np.array([char.decode('utf-8').strip() for char in dr['CopyMetaData'].values])
    namelist         = np.array([char.decode('utf-8').strip() for char in dr['namelist'].values])
    time             = dr['time'].values
    ObsTypes         = dr['ObsTypes'].values        
    ObsIndex         = dr['ObsIndex'].values     
    obs_type         = dr['obs_type'].values
    obs_keys         = dr['obs_keys'].values
    which_vert       = dr['which_vert'].values
    location         = dr['location'].values
    observations     = dr['observations'].values
    qc               = dr['qc'].values
    
    
    ind_qc  = np.where(QCMetaData == var_dict['QCString'])
    reg_xyz = np.array(var_dict['region'].split(' '), dtype=float)
    
    print(min(location[:,0]),max(location[:,0]))
    print(min(location[:,1]),max(location[:,1]))
    print(min(location[:,2]),max(location[:,2]))
    print(reg_xyz[0],reg_xyz[1])
    print(reg_xyz[2],reg_xyz[3])
    print(reg_xyz[4],reg_xyz[5])
    
    ind_loc = np.where(location[:,0] >= reg_xyz[0] and location[:,0] <= reg_xyz[1] and 
                       location[:,1] >= reg_xyz[2] and location[:,1] <= reg_xyz[3] and 
                       location[:,2] >= reg_xyz[4] and location[:,2] <= reg_xyz[5])
    print(location[ind_loc])
    print(reg_latlon)
    print(xxx)
    print(qc[:,ind_qc])
    print(xxx)
    print(QCMetaData[ind_qc])
    print(xxxx)
    for qctype in QCMetaData: 
        print(str(qctype))
    
    print(QCMetaData.size)
    print(time.size)
    #print(np.array(str(QCMetaData)))
    #print(qc)
    print(xxx)

    
    return(dart_obs)


In [127]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = os.path.join(top_path,"paper_material","case_study","figure_data")
    fig_path = os.path.join(top_path,"paper_material","case_study","figures","mean_series")
    
    var_dict = {'all': {'ObsTypeString': 'all',
                        'CopyString'   : 'observation', 
                        'QCString'     : 'DART quality control',  
                        'nlev'         : 21, 
                        'region'       : '0 360 -90 90 0 120000',
                        'verbose'      : False,
                       },
    }
    
    exp_dict = extract_exp_info()
    
    case_name = 'PNWSNOW'
    resolution = "F20TR_ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    diag_key = "obs_seq_final"
    frequency = "6hourly"
    
    path_template  = "/compyfs/zhan391/v3_dart_cda_scratch/%(RUNNAME)/archive/%(CASENAME)/dart_diagnostics/%(DIAG)"
    file_template  = "%(RUNNAME)_%(RES)_%(MACH).dart.e.eam_%(KEY).%(TIME)-*.nc"
    
    date = "2011-12-02"
    var = 'all'
    for exp in exp_dict.keys():
        run = exp_dict[exp]['run']
        key = exp_dict[exp]['key']
        diag = exp_dict[exp]['diag1']
        path = path_template.replace('%(RUNNAME)',run).replace('%(CASENAME)',key).replace('%(DIAG)',diag)
        file = file_template.replace('%(RUNNAME)',exp).replace('%(RES)',resolution).replace('%(MACH)',machine).replace('%(KEY)',diag_key).replace('%(TIME)',date)
        read_dart_obs_diag(var,var_dict[var],date,path,file)
                

0.0 359.99728666398096
-89.99999999999899 88.86919139983567
-23.0 103210.0
0.0 360.0
-90.0 90.0
0.0 120000.0


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()